In [1]:
pip install yfinance

Note: you may need to restart the kernel to use updated packages.


In [2]:
import yfinance as yf
import pandas as pd
import numpy as np 

In [3]:
data = yf.download("XU100.IS", start="2019-01-01", end="2026-07-03")

[*********************100%***********************]  1 of 1 completed


In [4]:
data['log_return'] = np.log(data['Close'] / data['Close'].shift(1))
data['volatility_10d'] = data['log_return'].rolling(window=10).std()

In [5]:
data.dropna(inplace=True)
print(data[['Close', 'log_return', 'volatility_10d']].head(15))

Price             Close log_return volatility_10d
Ticker         XU100.IS                          
Date                                             
2019-01-16   954.109924   0.023738       0.010754
2019-01-17   968.168884   0.014628       0.006960
2019-01-18   984.543762   0.016772       0.007012
2019-01-21   979.546814  -0.005088       0.008451
2019-01-22   996.765686   0.017426       0.008904
2019-01-23  1001.404663   0.004643       0.008928
2019-01-24  1017.737610   0.016178       0.008551
2019-01-25  1018.008606   0.000266       0.009060
2019-01-28  1012.892639  -0.005038       0.010160
2019-01-29  1040.978516   0.027351       0.011623
2019-01-30  1041.889526   0.000875       0.011094
2019-01-31  1040.736450  -0.001107       0.011290
2019-02-01  1029.365479  -0.010986       0.012069
2019-02-04  1022.544556  -0.006648       0.012215
2019-02-05  1024.478516   0.001890       0.011314


In [6]:
print(data[['Close', 'log_return', 'volatility_10d']].tail(15))

Price              Close log_return volatility_10d
Ticker          XU100.IS                          
Date                                              
2026-06-12  13938.500000   0.014089       0.015427
2026-06-15  14446.400391   0.035790       0.018784
2026-06-16  14493.099609   0.003227       0.015482
2026-06-17  14421.200195  -0.004973       0.014312
2026-06-18  14827.400391   0.027777       0.015740
2026-06-19  14734.500000  -0.006285       0.014942
2026-06-22  14729.700195  -0.000326       0.015018
2026-06-23  14539.599609  -0.012990       0.015550
2026-06-24  14331.200195  -0.014437       0.016761
2026-06-25  14259.799805  -0.004995       0.016970
2026-06-26  14274.000000   0.000995       0.016579
2026-06-29  14183.200195  -0.006382       0.011815
2026-06-30  14121.799805  -0.004338       0.011696
2026-07-01  14350.599609   0.016072       0.013037
2026-07-02  14455.000000   0.007249       0.009119


In [7]:
data.isnull().sum()

Price           Ticker  
Close           XU100.IS    0
High            XU100.IS    0
Low             XU100.IS    0
Open            XU100.IS    0
Volume          XU100.IS    0
log_return                  0
volatility_10d              0
dtype: int64

In [8]:
data['target_volatility'] = data['volatility_10d'].shift(-10)

In [9]:
data.isnull().sum()

Price              Ticker  
Close              XU100.IS     0
High               XU100.IS     0
Low                XU100.IS     0
Open               XU100.IS     0
Volume             XU100.IS     0
log_return                      0
volatility_10d                  0
target_volatility              10
dtype: int64

In [10]:
data = data.dropna()
data = data.copy() 

In [11]:
data.isnull().sum()

Price              Ticker  
Close              XU100.IS    0
High               XU100.IS    0
Low                XU100.IS    0
Open               XU100.IS    0
Volume             XU100.IS    0
log_return                     0
volatility_10d                 0
target_volatility              0
dtype: int64

In [12]:
data['volatility_lag1'] = data['volatility_10d'].shift(1)
data['volatility_lag5'] = data['volatility_10d'].shift(5)

In [13]:
data['ma_5'] = data['Close'].rolling(window=5).mean()
data['ma_20'] = data['Close'].rolling(window=20).mean()

In [14]:
data['volume_ma_5'] = data['Volume'].rolling(window=5).mean()

In [15]:
delta = data['Close'].diff()
gain = delta.where(delta > 0, 0)
loss = -delta.where(delta < 0, 0)

avg_gain = gain.rolling(window=14).mean()
avg_loss = loss.rolling(window=14).mean()

rs = avg_gain / avg_loss
data['rsi'] = 100 - (100 / (1 + rs))

In [16]:
data[['Close', 'rsi']].tail(10)

Price,Close,rsi
Ticker,XU100.IS,
Date,,
2026-06-05,13694.200195,35.355691
2026-06-08,13860.599609,40.006496
2026-06-09,13741.900391,38.008714
2026-06-10,13744.599609,41.074494
2026-06-11,13743.500000,45.464634
2026-06-12,13938.500000,48.896561
2026-06-15,14446.400391,71.444345
2026-06-16,14493.099609,64.312281


In [17]:
data = data.dropna()
data = data.copy() 

In [18]:
split_index = int(len(data) * 0.8)

train = data.iloc[:split_index]
test = data.iloc[split_index:]

features = ['volatility_lag1', 'volatility_lag5', 'ma_5', 'ma_20', 'volume_ma_5', 'rsi']

X_train = train[features]
y_train = train['target_volatility']

X_test = test[features]
y_test = test['target_volatility']

In [19]:
X_train.shape

(1464, 6)

In [20]:
X_test.shape

(366, 6)

In [21]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
mae_lr = mean_absolute_error(y_test, y_pred_lr)

print(f"Linear Regression - RMSE: {rmse_lr:.5f}, MAE: {mae_lr:.5f}")

Linear Regression - RMSE: 0.00729, MAE: 0.00591


In [22]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
mae_rf = mean_absolute_error(y_test, y_pred_rf)

print(f"Random Forest - RMSE: {rmse_rf:.5f}, MAE: {mae_rf:.5f}")

Random Forest - RMSE: 0.00831, MAE: 0.00669


In [23]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr_scaled = LinearRegression()
lr_scaled.fit(X_train_scaled, y_train)

coefficients_scaled = pd.Series(lr_scaled.coef_, index=features)
print(coefficients_scaled.sort_values(ascending=False))

ma_5               0.007913
volatility_lag1    0.001161
volatility_lag5    0.000768
volume_ma_5       -0.000120
rsi               -0.000411
ma_20             -0.007175
dtype: float64


In [24]:
X_train[features].corr()

,Price,volatility_lag1,volatility_lag5,ma_5,ma_20,volume_ma_5,rsi
,Ticker,,,,,,
Price,Ticker,,,,,,
volatility_lag1,,1.000000,0.730453,0.126274,0.137352,-0.156112,-0.341676
volatility_lag5,,0.730453,1.000000,0.135911,0.139732,-0.170685,-0.201777
ma_5,,0.126274,0.135911,1.000000,0.998627,-0.421159,-0.009435
ma_20,,0.137352,0.139732,0.998627,1.000000,-0.419126,-0.045001
volume_ma_5,,-0.156112,-0.170685,-0.421159,-0.419126,1.000000,0.021413
rsi,,-0.341676,-0.201777,-0.009435,-0.045001,0.021413,1.000000


In [25]:
features_v2 = ['volatility_lag1', 'volatility_lag5', 'ma_5', 'volume_ma_5', 'rsi']

X_train_v2 = train[features_v2]
X_test_v2 = test[features_v2]

lr_v2 = LinearRegression()
lr_v2.fit(X_train_v2, y_train)

y_pred_v2 = lr_v2.predict(X_test_v2)
rmse_v2 = np.sqrt(mean_squared_error(y_test, y_pred_v2))
print(f"RMSE (ma_20 çıkarıldı): {rmse_v2:.5f}")

RMSE (ma_20 çıkarıldı): 0.00737


In [26]:
scaler_v2 = StandardScaler()
X_train_v2_scaled = scaler_v2.fit_transform(X_train_v2)
X_test_v2_scaled = scaler_v2.transform(X_test_v2)

lr_v2_scaled = LinearRegression()
lr_v2_scaled.fit(X_train_v2_scaled, y_train)

coefficients_v2 = pd.Series(lr_v2_scaled.coef_, index=features_v2)
print(coefficients_v2.sort_values(ascending=False))

volatility_lag1    0.001139
volatility_lag5    0.000805
ma_5               0.000742
volume_ma_5       -0.000135
rsi               -0.000155
dtype: float64


In [27]:
data.tail()

Price,Close,High,Low,Open,Volume,log_return,volatility_10d,target_volatility,volatility_lag1,volatility_lag5,ma_5,ma_20,volume_ma_5,rsi
Ticker,XU100.IS,XU100.IS,XU100.IS,XU100.IS,XU100.IS,,,,,,,,,
Date,,,,,,,,,,,,,,
2026-06-12,13938.500000,14125.799805,13801.299805,13912.099609,9534100000,0.014089,0.015427,0.016579,0.015798,0.030181,13805.819922,13971.175049,9.339460e+09,48.896561
2026-06-15,14446.400391,14508.599609,14251.500000,14359.500000,8754000000,0.035790,0.018784,0.011815,0.015427,0.030530,13922.980078,13954.500049,9.234040e+09,71.444345
2026-06-16,14493.099609,14575.299805,14417.900391,14482.900391,9631500000,0.003227,0.015482,0.011696,0.018784,0.022090,14073.219922,13949.230029,9.452400e+09,64.312281
2026-06-17,14421.200195,14604.599609,14375.400391,14583.200195,9958800000,-0.004973,0.014312,0.013037,0.015482,0.015955,14208.540039,13938.055029,1.007496e+10,61.131877
2026-06-18,14827.400391,14827.400391,14454.299805,14454.299805,9232300000,0.027777,0.015740,0.009119,0.014312,0.015798,14425.320117,13961.045068,9.422140e+09,72.746131


In [28]:
raw = yf.download("XU100.IS", start="2019-01-01", end="2026-07-03")

raw['log_return'] = np.log(raw['Close'] / raw['Close'].shift(1))
raw['volatility_10d'] = raw['log_return'].rolling(window=10).std()

raw['volatility_lag1'] = raw['volatility_10d'].shift(1)
raw['volatility_lag5'] = raw['volatility_10d'].shift(5)
raw['ma_5'] = raw['Close'].rolling(window=5).mean()
raw['volume_ma_5'] = raw['Volume'].rolling(window=5).mean()

delta = raw['Close'].diff()
gain = delta.where(delta > 0, 0)
loss = -delta.where(delta < 0, 0)
avg_gain = gain.rolling(window=14).mean()
avg_loss = loss.rolling(window=14).mean()
rs = avg_gain / avg_loss
raw['rsi'] = 100 - (100 / (1 + rs))

latest_row = raw[features_v2].iloc[[-1]]
print(latest_row)

prediction = lr_v2.predict(latest_row)
print(f"Önümüzdeki 10 günün tahmini volatilitesi: {prediction[0]:.5f}")

[*********************100%***********************]  1 of 1 completed

Price      volatility_lag1 volatility_lag5          ma_5   volume_ma_5  \
Ticker                                                                   
Date                                                                     
2026-07-02        0.013037         0.01697  14276.919922  6.981500e+09   

Price             rsi  
Ticker                 
Date                   
2026-07-02  62.298202  
Önümüzdeki 10 günün tahmini volatilitesi: 0.01841


In [29]:
print(f"Son 10 günün gerçekleşen volatilitesi: {data['volatility_10d'].iloc[-1]:.5f}")
print(f"Son 1 yılın ortalama volatilitesi: {data['volatility_10d'].tail(252).mean():.5f}")
print(f"Tahmin edilen önümüzdeki 10 günün volatilitesi: {prediction[0]:.5f}")

Son 10 günün gerçekleşen volatilitesi: 0.01574
Son 1 yılın ortalama volatilitesi: 0.01398
Tahmin edilen önümüzdeki 10 günün volatilitesi: 0.01841
